In [ ]:
# Mount Google Drive

# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import json

with open('/path/to/your/file', 'r') as f:
    merged_data = json.load(f)

In [ ]:
print(merged_data[0:1])

In [ ]:
def remove_outside_tokens(dataset):
    filtered_data = []

    for example in dataset:
        filtered_example = {
            'id': example['id'],
            'tokens': [],
            'ner_tags_1': [],
            'ner_tags_2': []
        }

        for token, label_1, label_2 in zip(example['tokens'], example['ner_tags_1'], example['ner_tags_2']):
            if label_1 != 0 or label_2 != 0:
                filtered_example['tokens'].append(token)
                filtered_example['ner_tags_1'].append(label_1)
                filtered_example['ner_tags_2'].append(label_2)

        if filtered_example["tokens"]:
            filtered_data.append(filtered_example)

    return filtered_data

In [ ]:
filtered_dataset = remove_outside_tokens(merged_data)

In [ ]:
def count_o_labels(dataset):
    o_count_1 = sum(1 for example in dataset for label in example["ner_tags_1"] if label == 0)
    o_count_2 = sum(1 for example in dataset for label in example["ner_tags_2"] if label == 0)

    print(f"O labels in ner_tags_1: {o_count_1}")
    print(f"O labels in ner_tags_2: {o_count_2}")

print("Before filtering:")
count_o_labels(merged_data)

print("After filtering:")
count_o_labels(filtered_dataset)

In [ ]:
#print ner_tags_2
for example in filtered_dataset:
    print(example["ner_tags_2"])

In [ ]:
print(filtered_dataset[0])

In [ ]:
from openai import OpenAI
import os

# Set your OpenAI API key
api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(
  api_key=api_key
)

from tqdm import tqdm
import time

In [ ]:
# Mapping for conversion
id2label = {1: "Implicature", 2: "Ambiguity", 3: "Presupposition", 4: "Explicit"}
label2id = {"Implicature": 1, "Ambiguity": 2, "Presupposition": 3, "Explicit": 4}

In [ ]:
filtered_data = []
for example in filtered_dataset:
    filtered_data.append({
        "id": example["id"],
        "tokens": example["tokens"],
        "labels": [id2label[label] for label in example["ner_tags_2"]]  # Convert to text labels
    })

In [ ]:
print(filtered_data[0])

In [ ]:
unique_labels = set()
for example in filtered_data:
    unique_labels.update(example["labels"])

print(f"Unique labels in labels: {unique_labels}")

In [ ]:
print(type(list))  # Should be <class 'type'>

In [ ]:
#Convert a list of tokens to sentences
def tokens_to_text(tokens):
    """
    Converts a list of tokens into a readable text document.
    Ensures proper spacing around punctuation.
    """
    text = ""
    for token in tokens:
        if token in {".", ",", "!", "?", ":", ";", "'s", "'ve", "'re", "'ll", "'d", "'m", "n't"}:
            text += token
        else:
            text += " " + token

    return text.strip()

In [ ]:
text21 = tokens_to_text(filtered_data[21]["tokens"])
print("🔍 Full Text of Document 0:\n")
print(text21)

In [ ]:
def classify_sentences(text):

    prompt = f"""
    You are an AI performing sentence classification.
    I will provide a full document, and you will wrap each sentence in either <Explicit> or <Implicature> or <Ambiguity> or <Presupposition> tags.
    Explicit refers to clear and fully comprehensive statements.
    Implicature, ambiguity and presupposition are fine-grained versions of implicit, meaning that everything that has implicit meaning has to be categorized into one of these three labels.
    Implicature itself refers to content which represents universal knowledge or which is inferable from the utterance and its context, though not explicitly expressed.
    Ambiguity itself refers to content that has the potential to denote multiple entities, requiring precision due to its inability to be inferred from the context alone.
    Presupposition itself consists in presenting some notion as already shared by the author and their addressee(s).
    DO NOT CHANGE THE INPUT TEXT. DO NOT CORRECT THE INPUT TEXT, DO NOT MISTAKES.

    Example Input:
    People say that politicians lie all the time.

    Example Output:
    <Implicature> People say that politicians lie all the time. </Implicature>

    Now classify this text:
    {text}
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=2048
        )

        output = response.choices[0].message.content.strip()

        if not output:
            print("Warning: Empty response received. Retrying...")
            time.sleep(2)
            return classify_sentences(text)

        return output

    except Exception as e:
        print(f"API Error: {e}")
        return None

In [ ]:
annotated_text = classify_sentences(text21)
print("\n🔍 Annotated Document:\n", annotated_text)

Iterate over all doc


In [ ]:
import json
import time

output_file = "/path/to/your/file"
annotated_data = []

for i in range(78):
    text = tokens_to_text(filtered_data[i]["tokens"])
    classification_result = classify_sentences(text)

    if classification_result:  # Ensure it's not None
        annotated_data.append({
            "id": filtered_data[i]["id"],
            "annotated_text": classification_result
        })

        # Save after each document
        with open(output_file, "w") as f:
            json.dump(annotated_data, f, indent=4)

        print(f"Saved classification for document {i}/{len(filtered_data)}")

    else:
        print(f"Skipped document {i} due to invalid response.")

    time.sleep(1)

print(f"\n Annotations saved to {output_file}")

In [ ]:
import json
import re
import nltk

nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

In [ ]:
# Load GPT-generated annotations
gpt_output_file = "/path/to/your/file"
with open(gpt_output_file, "r") as f:
    gpt_annotations = json.load(f)

In [ ]:
def tokenize_text(text):
    text = text.replace("<Explicit>", "").replace("</Explicit>", "").replace("</Explicit", "")
    text = text.replace("<Implicature>", "").replace("</Implicature>", "")
    text = text.replace("<Ambiguity>", "").replace("</Ambiguity>", "")
    text = text.replace("<Presupposition>", "").replace("</Presupposition>", "")

    tokens = re.findall(r"\w+|[^\w\s]", text)
    return tokens

In [ ]:
def parse_annotated_gpt_text(raw_text):
    """
    Splits out tokens and assigns 'Explicit'/'Implicit'/'O'.
    """
    tag_pattern = re.compile(r"(</?Explicit>|</?Implicature>|</?Ambiguity>|</?Presupposition>)")
    parts = tag_pattern.split(raw_text)

    tokens = []
    labels = []
    current_label = "O"

    for part in parts:
        part = part.strip()
        if not part:
            continue

        if part == "<Explicit>":
            current_label = "Explicit"
            continue
        elif part == "</Explicit>" or part == "</Explicit":
            current_label = "Explicit"
            continue
        elif part == "<Implicature>":
            current_label = "Implicature"
            continue
        elif part == "</Implicature>":
            current_label = "Implicature"
            continue
        elif part == "<Ambiguity>":
            current_label = "Ambiguity"
            continue
        elif part == "</Ambiguity>":
            current_label = "Ambiguity"
            continue
        elif part == "<Presupposition>":
            current_label = "Presupposition"
            continue
        elif part == "</Presupposition>":
            current_label = "Presupposition"
            continue
        else:
            sub_tokens = re.findall(r"\w+|[^\w\s]", part)
            for st in sub_tokens:
                tokens.append(st)
                labels.append(current_label)

    return tokens, labels

In [ ]:
def align_tokens(tokens):

    contraction_bases = {
        "do": "don", "wo": "won", "ca": "can", "sha": "shan", "is": "isn",
        "are": "aren", "was": "wasn", "were": "weren", "have": "haven",
        "has": "hasn", "had": "hadn", "should": "shouldn", "would": "wouldn",
        "could": "couldn", "might": "mightn", "must": "mustn", "does": "doesn",
        "did": "didn",
    }

    aligned_tokens = []
    i = 0
    while i < len(tokens):
        lower_tok = tokens[i].lower()

        if (
            i + 3 < len(tokens) and
            lower_tok in contraction_bases and
            tokens[i+1].lower() == "n" and
            tokens[i+2] == "'" and
            tokens[i+3].lower() == "t"
        ):
            merged = contraction_bases[lower_tok]
            aligned_tokens.append(merged)
            aligned_tokens.append("'")
            aligned_tokens.append("t")
            i += 4
        else:
            aligned_tokens.append(tokens[i])
            i += 1

    return aligned_tokens

In [ ]:
def align_tokens_with_labels(tokens, labels):
    contraction_bases = {
        "do": "don", "wo": "won", "ca": "can", "sha": "shan", "is": "isn",
        "are": "aren", "was": "wasn", "were": "weren", "have": "haven",
        "has": "hasn", "had": "hadn", "should": "shouldn", "would": "wouldn",
        "could": "couldn", "might": "mightn", "must": "mustn", "does": "doesn",
        "did": "didn"
    }

    aligned_tokens = []
    aligned_labels = []
    i = 0
    while i < len(tokens):
        lower_tok = tokens[i].lower()

        if (
            i + 3 < len(tokens) and
            lower_tok in contraction_bases and
            tokens[i+1].lower() == "n" and
            tokens[i+2] == "'" and
            tokens[i+3].lower() == "t"
        ):
            # Merge base + "n"
            merged_token = contraction_bases[lower_tok]
            merged_label = labels[i]

            aligned_tokens.append(merged_token)
            aligned_labels.append(merged_label)

            # add apostrophe, 't'
            aligned_tokens.append("'")
            aligned_labels.append(labels[i+2])

            aligned_tokens.append("t")
            aligned_labels.append(labels[i+3])

            i += 4
        else:
            aligned_tokens.append(tokens[i])
            aligned_labels.append(labels[i])
            i += 1

    return aligned_tokens, aligned_labels

In [ ]:
import re

N = len(filtered_data)
entries_to_fix = []
aligned_data = []
gpt_aligned_data = []

for e in range(N):
    new_tokens = []
    new_labels = []

    original_tokens = filtered_data[e]['tokens']
    original_labels = filtered_data[e]['labels']

    for token, label in zip(original_tokens, original_labels):
        sub_tokens = re.findall(r"\w+|[^\w\s]", token)
        for st in sub_tokens:
            new_tokens.append(st)
            new_labels.append(label)

    gold_pred_tokens, gold_pred_labels = align_tokens_with_labels(new_tokens, new_labels)

    gpt_tokens, gpt_labels = parse_annotated_gpt_text(gpt_annotations[e]['annotated_text'])
    aligned_gpt_tokens, aligned_gpt_labels = align_tokens_with_labels(gpt_tokens, gpt_labels)

    if e in entries_to_fix:
        if len(gold_pred_tokens) > len(aligned_gpt_tokens):
            max_len = len(aligned_gpt_tokens)
            gold_pred_tokens = gold_pred_tokens[:max_len]
            gold_pred_labels = gold_pred_labels[:max_len]

        if len(aligned_gpt_tokens) > len(gold_pred_tokens):
            max_len = len(gold_pred_tokens)
            aligned_gpt_tokens = aligned_gpt_tokens[:max_len]
            aligned_gpt_labels = aligned_gpt_labels[:max_len]

    aligned_data.append({
        'id': filtered_data[e]['id'],
        'tokens': gold_pred_tokens,
        'labels': gold_pred_labels
    })

    gpt_aligned_data.append({
        'id': gpt_annotations[e]['id'],
        'tokens': aligned_gpt_tokens,
        'labels': aligned_gpt_labels
    })

    print(f"Entry {e} processed:")
    print("gold_pred_tokens length:", len(gold_pred_tokens))
    print("aligned_gpt_tokens length:", len(aligned_gpt_tokens))

print("\nDone processing all entries")
print("aligned_data length:", len(aligned_data))
print("gpt_aligned_data length:", len(gpt_aligned_data))

In [ ]:
print(aligned_data[24])
print(gpt_aligned_data[24])
print(len(aligned_data[24]['tokens']))
print(len(gpt_aligned_data[24]['tokens']))

In [ ]:
from sklearn.metrics import classification_report

excluded_entries = []

all_gold_labels = []
all_pred_labels = []

for e in range(len(aligned_data)):
    if e in excluded_entries:
        continue

    # gold labels
    gold_labels = aligned_data[e]['labels']
    # predicted labels (GPT)
    pred_labels = gpt_aligned_data[e]['labels']

    all_gold_labels.extend(gold_labels)
    all_pred_labels.extend(pred_labels)

print("Evaluating on all entries except:", excluded_entries)
print(classification_report(all_gold_labels, all_pred_labels))

___________________________________________________
___________________________________________________